# Injection-recovery of HARPS-N − SunSim: our criterion and the criteria of the other studies

**Setup (as in the paper, Ignasi's choice).** The instrumental offsets (one per HARPS-N intervention interval, 21) and,
if REMOVE_860, the ≈860 d sinusoid are fitted **once** to the series, without any planet, and removed. The planets are
then injected in these residuals and searched with a plain GLS (one constant + the sinusoid), **without any offset**.

**Assumption.** The instrument stays untouched during the whole monitoring: no reset, so no offset can absorb part of
the planet. This is optimistic, but it is the situation needed to find Sun–Earth analogues: periods longer than the
time between two resets of the instrument are lost (with an offset every ~5 months, as the 21 HARPS-N intervals, the
limits at P ≳ 1 yr are clearly worse). For the paper: state this assumption explicitly.

**The ≈860 d signal.** Dumusque et al. (2026) find signals left at ≈895 and ≈585 d in the HARPS-N solar RVs and the
teams of the PLATO RV challenge (Cretignier et al. 2026) find spurious long-period signals in the same data. The check
below compares its amplitude in HARPS-N − SunSim, HARPS-N alone and SunSim alone: if it is in HARPS-N and not in
SunSim, it is in the observations (instrumental, or solar and not modelled).

Sections: our criterion (empirical FAP, highest peak within 10 % of the injected period), then the criteria of each
study as described in its text:

| | data | injections | detection |
|---|---|---|---|
| Zhao et al. (2024) | HARPS-N 2015–2024, daily | P log-spaced 10–550 d, K 0.1–0.5 m/s (step 0.033), one injection per (P, K) | power at the injected period > 0.1 % FAP ("numerically estimated"); limit = lowest K detected |
| Colwell et al. (2025) | HARPS-N 2015–2018 (3 yr), exposures | P = 50 d, K = 0.1–1.0 m/s; P = 10–250 d at K = 1 m/s | period of the highest peak and amplitude fitted, compared with the injected ones (no FAP) |
| Gómez-Vargas et al. (2026) | HARPS-N 2015–2024, daily | P 10–550 d, 10 realisations | power within ±5 % of the injected period > 0.1 % FAP in ≥ 70 % of the realisations |
| Liang et al. (2026) | NEID 2020–2024, 1294 d | 500 trials, P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase | highest candidate, threshold raised until no false positive among the 500 trials; recovered if its period is within 20 % and \|K_rec − K\|/K < 1 |

Choices where the papers do not say: the 0.1 % FAP of Zhao is a bootstrap (values shuffled within the intervals), the
period tolerance of Zhao is ±5 % (as Gómez-Vargas), the FAP of Gómez-Vargas is analytic, the success of Colwell is the
highest peak within 5 % of the period and the amplitude within 20 %.

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt

from sunsim import local_path, observations as obs
from sunsim.residuals import ResidualAnalysis

SIM_RV = os.path.join(local_path('personal_data'), 'rv-15.txt')
ORIGIN = '2015-07-18'
FIRST, LAST = '2015-07-18', '2024-01-28'
FIGURES = "/Users/sophiestucki/Documents/rv_opt/figures/final/"
REMOVE_860 = True                    # also remove the ~860 d sinusoid (fitted once with the offsets)
FAP = 1e-3
SEED = 1
N_NULL = 2000                        # noise versions for the empirical FAPs
N_PHASES = 100                       # phases per (P, K) where a fraction of detections is computed
PERIODS = np.sort(np.append(np.linspace(5, 500, 10), 365.))          # our criterion: injected periods [d]
AMPLITUDES = np.round(np.arange(0.10, 0.501, 0.02), 3)               # our criterion: injected K [m/s]
PERIOD_TOL = 0.1                     # our criterion: highest peak within 10 % of the injected period


def save(fig, name):
    if FIGURES:
        fig.savefig(os.path.join(FIGURES, name + '.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
t_s, rv_s = np.loadtxt(SIM_RV).T
window = (t_s >= obs.jd_of_date(FIRST) - obs.jd_of_date(ORIGIN)) & (t_s < obs.jd_of_date(LAST) - obs.jd_of_date(ORIGIN) + 1)
t_s, rv_s = t_s[window], rv_s[window]
harps = obs.load_harpsn(local_path('harpsn_sun'), ORIGIN, end=LAST, t_start=FIRST)
cuts = obs.load_interventions(local_path('harpsn_interventions'), ORIGIN)
rv = obs.compare_rv(harps, t_s, rv_s, interventions=cuts)
obs.print_summary(rv=rv)

## Residuals: offsets (and the ≈860 d sinusoid) fitted once and removed

In [ ]:
def design(t, groups=None, periods=(), poly=0):
    """Columns: one indicator per group (or a constant), t^1..t^poly (t scaled to [-1, 1]), cos and sin of every period."""
    cols = []
    if groups is None:
        cols.append(np.ones(len(t)))
    else:
        g = np.unique(groups, return_inverse=True)[1]
        cols += [(g == k).astype(float) for k in range(g.max() + 1)]
    x = 2 * (t - t.min()) / np.ptp(t) - 1
    cols += [x ** p for p in range(1, poly + 1)]
    for P in periods:
        cols += [np.cos(2 * np.pi * t / P), np.sin(2 * np.pi * t / P)]
    return np.column_stack(cols)


def fit(t, y, **kw):
    """Least-squares fit of design(t, **kw): returns the residuals and the coefficients."""
    X = design(t, **kw)
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return y - X @ beta, beta


def best_long_period(t, y, pmin=500, pmax=1500, **kw):
    """Period in [pmin, pmax] d of the sinusoid that, fitted with design(**kw), lowers the residuals most, and its
    semi-amplitude."""
    T = np.ptp(t)
    freqs = np.arange(1 / pmax, 1 / pmin, 1 / (10 * T))
    rss = [np.sum(fit(t, y, periods=(1 / f,), **kw)[0] ** 2) for f in freqs]
    P = 1 / freqs[int(np.argmin(rss))]
    beta = fit(t, y, periods=(P,), **kw)[1]
    return P, float(np.hypot(beta[-2], beta[-1]))


order = np.argsort(rv['t'], kind='stable')
T = np.asarray(rv['t'], dtype=float)[order]
G = np.asarray(rv['period'])[order]
SERIES = {'res': np.asarray(rv['res'], dtype=float)[order], 'obs': np.asarray(rv['obs'], dtype=float)[order],
          'sim': np.asarray(rv['sim'], dtype=float)[order]}

# is the long-period signal in the observations or in the model?
P_NUISANCE, A_res = best_long_period(T, SERIES['res'], groups=G)
for name, y, kw in (('HARPS-N - SunSim (21 offsets)', SERIES['res'], dict(groups=G)),
                    ('HARPS-N alone (21 offsets, cycle as a cubic)', SERIES['obs'], dict(groups=G, poly=3)),
                    ('SunSim alone (cycle as a cubic)', SERIES['sim'], dict(poly=3))):
    P, A = best_long_period(T, y, **kw)
    A_fixed = np.hypot(*fit(T, y, periods=(P_NUISANCE,), **kw)[1][-2:])
    print('%-46s best 500-1500 d: %4.0f d, %.2f m/s;  at %.0f d: %.2f m/s' % (name, P, A, P_NUISANCE, A_fixed))

models = {}
for key, label, kw in (('res', 'HARPS-N - SunSim', dict(groups=G)), ('obs', 'HARPS-N', dict(groups=G, poly=0))):
    y = fit(T, SERIES[key], periods=(P_NUISANCE,) if REMOVE_860 else (), **kw)[0]
    models[key] = ResidualAnalysis(T, y, groups=G,
                                   name=label + (' (offsets + %.0f d removed)' % P_NUISANCE if REMOVE_860 else ' (offsets removed)'))
    print('%-50s %d points, rms %.2f m/s' % (models[key].name, models[key].N, np.std(y)))

## Tools

In [ ]:
def injected(ra, P, K, t0):
    """ra.y + K sin(2 pi (t - t0) / P), one column per (P, K, t0) (arrays of the same length or scalars)."""
    P, K, t0 = np.broadcast_arrays(np.atleast_1d(P), np.atleast_1d(K), np.atleast_1d(t0))
    return ra.y[:, None] + K[None] * np.sin(2 * np.pi * (ra.t[:, None] - t0[None]) / P[None])


def analyse(ra, Y, P_inj, tol=0.05, chunk=200):
    """For every column of Y: period, power and semi-amplitude of the highest peak, and the highest power within
    ±tol of the injected period P_inj (one per column)."""
    P_inj = np.broadcast_to(np.atleast_1d(P_inj), (Y.shape[1],))
    out = {k: np.empty(Y.shape[1]) for k in ('P', 'pmax', 'K', 'local')}
    for i in range(0, Y.shape[1], chunk):
        sl = slice(i, i + chunk)
        power, a, b, _ = ra._gls(Y[:, sl])
        k = power.argmax(axis=0)
        c = np.arange(power.shape[1])
        out['P'][sl], out['pmax'][sl], out['K'][sl] = 1 / ra.freq[k], power[k, c], np.hypot(a[k, c], b[k, c])
        f = ra.freq[:, None]
        near = (f >= 1 / ((1 + tol) * P_inj[sl])) & (f <= 1 / ((1 - tol) * P_inj[sl]))
        out['local'][sl] = np.where(near, power, -np.inf).max(axis=0)
    return out


def null_peaks(ra, method='shift', n=N_NULL, seed=SEED):
    """Highest peak (power, period) of n noise-only versions of ra.y: 'shuffle' (values permuted inside every
    interval: bootstrap) or 'shift' (series rolled cyclically: keeps the correlated noise)."""
    rng = np.random.default_rng(seed)
    if method == 'shuffle':
        Y = np.repeat(ra.y[:, None], n, axis=1)
        for g in np.unique(ra.groups):
            idx = np.flatnonzero(ra.groups == g)
            Y[idx] = rng.permuted(Y[idx], axis=0)
    else:
        k = rng.integers(int(0.05 * ra.N), int(0.95 * ra.N), size=n)
        Y = np.stack([np.roll(ra.y, s) for s in k], axis=1)
    r = analyse(ra, Y, np.ones(n))
    return {'pmax': r['pmax'], 'P': r['P']}


def null_level(ra, method='shuffle', fap=FAP):
    """Power of the highest peak exceeded by a fraction fap of the noise-only versions."""
    return np.quantile(null_peaks(ra, method)['pmax'], 1 - fap)


def limit(K, rate, level):
    """Lowest K where rate [%] (along K) first reaches level (linear interpolation; NaN: never)."""
    above = np.flatnonzero(rate >= level)
    if len(above) == 0:
        return np.nan
    i = above[0]
    return K[0] if i == 0 else float(np.interp(level, rate[i - 1:i + 1], K[i - 1:i + 1]))


def limit_all(K, rate):
    """Smallest K above which every K of the grid is recovered at 100 % (NaN: not within the grid)."""
    bad = np.flatnonzero(rate < 100)
    return K[0] if len(bad) == 0 else (K[bad[-1] + 1] if bad[-1] < len(K) - 1 else np.nan)


def msini(K, P):
    """M sin i [Earth masses] for K [m/s], P [d], 1 Msun, circular orbit."""
    k_earth = (2 * np.pi * 6.674e-11 / (365.25 * 86400)) ** (1 / 3) * 5.972e24 / 1.989e30 ** (2 / 3)
    return np.asarray(K) / k_earth * (np.asarray(P) / 365.25) ** (1 / 3)

summary = {}

## Our criterion
Highest peak within 10 % of the injected period and above the empirical 0.1 % FAP (noise-only versions shifted
cyclically in time, which keeps the correlated noise). 100 phases per (P, K).

In [ ]:
rate_O, lim_O = {}, {}
for key, ra in models.items():
    null = null_peaks(ra, 'shift')
    thr = np.quantile(null['pmax'], 1 - FAP)
    rate = np.empty((len(AMPLITUDES), len(PERIODS)))
    for j, P in enumerate(PERIODS):
        t0 = ra.t[0] + P * np.arange(N_PHASES) / N_PHASES
        for i, K in enumerate(AMPLITUDES):
            r = analyse(ra, injected(ra, P, K, t0), P)
            rate[i, j] = 100 * np.mean((r['pmax'] >= thr) & (np.abs(r['P'] - P) / P <= PERIOD_TOL))
    rate_O[key] = rate
    lim_O[key] = {50: np.array([limit(AMPLITUDES, rate[:, j], 50) for j in range(len(PERIODS))]),
                  100: np.array([limit_all(AMPLITUDES, rate[:, j]) for j in range(len(PERIODS))])}
    fa = [100 * np.mean((null['pmax'] >= thr) & (np.abs(null['P'] - P) / P <= PERIOD_TOL)) for P in PERIODS]
    print('%-50s K at 50 %% / 100 %% [m/s]:' % ra.name)
    print('   ' + ' '.join('%3.0f d %.2f/%.2f' % (p, a, b) for p, a, b in zip(PERIODS, lim_O[key][50], lim_O[key][100])))
    print('   false alarms of the criterion [%%]: max %.2f' % max(fa))
i1 = int(np.argmin(np.abs(PERIODS - 365)))
l50 = lim_O['res'][50]
summary['Ours'] = ('-', '%.2f-%.2f m/s (5-500 d), %.2f at 1 yr = %.1f Earth masses (50 %%)'
                   % (np.nanmin(l50), np.nanmax(l50), l50[i1], msini(l50[i1], 365.25)))

## Zhao et al. (2024)
Grid of their paper (P log-spaced 10–550 d, K 0.1–0.5 m/s by 0.033), power within ±5 % of the injected period above the
bootstrap 0.1 % FAP. Their one injection per (P, K) is replaced by 100 phases: the map is the probability that one
injection is detected, and their "lowest K detected" corresponds to ~50 %.

In [ ]:
P_Z = np.geomspace(10, 550, 12)
K_Z = np.arange(0.1, 0.5 + 1e-9, 0.1 / 3)
rate_Z, fp_Z = {}, {}
for key, ra in models.items():
    thr = null_level(ra, 'shuffle')
    fp_Z[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])   # passes with no planet
    rate = np.empty((len(K_Z), len(P_Z)))
    for j, P in enumerate(P_Z):
        t0 = ra.t[0] + P * np.arange(N_PHASES) / N_PHASES
        for i, K in enumerate(K_Z):
            rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
    rate_Z[key] = rate
    lim = [limit(K_Z, rate[:, j], 50) for j in range(len(P_Z))]
    print('%-30s K at 50 %% (* = the series without planet already passes: not usable):' % ra.name)
    print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_Z[key])))
lims = np.array([limit(K_Z, rate_Z['res'][:, j], 50) for j in range(len(P_Z))])[~fp_Z['res']]
summary['Zhao et al. (2024)'] = ('0.18 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)' % (
    np.nanmin(lims), np.nanmax(lims), len(lims), len(P_Z)))

## Colwell et al. (2025)
First 3 years (2015-07-18 to 2018-07-18). Amplitude sweep at P = 50 d (K = 0.1–1.0 m/s) and period sweep at K = 1 m/s
(P = 10–250 d). Recovered: highest peak within 5 % of the period and fitted K within 20 %. Their result: 0.2 m/s at
50 d recovered with 8.8 % error on K and 0.7 % on P.

In [ ]:
t_end = obs.jd_of_date('2018-07-18') - obs.jd_of_date(ORIGIN)
m = models['res'].t <= t_end
ra_c = ResidualAnalysis(models['res'].t[m], models['res'].y[m], groups=models['res'].groups[m], name='HARPS-N - SunSim, 2015-2018')
print('%d days' % ra_c.N)
K_C = np.arange(0.1, 1.0 + 1e-9, 0.1)
t0 = ra_c.t[0] + 50 * np.arange(N_PHASES) / N_PHASES
print('P = 50 d:   K    recovered [%]   median |dK|/K   median |dP|/P')
rec_C = []
for K in K_C:
    r = analyse(ra_c, injected(ra_c, 50.0, K, t0), 50.0)
    ok = (np.abs(r['P'] - 50) / 50 <= 0.05) & (np.abs(r['K'] - K) / K <= 0.2)
    rec_C.append(100 * ok.mean())
    print('          %4.1f  %8.0f        %10.3f      %10.4f' % (K, 100 * ok.mean(), np.median(np.abs(r['K'] - K) / K),
                                                            np.median(np.abs(r['P'] - 50) / 50)))
print('P sweep at K = 1 m/s:', end='')
for P in (10, 25, 40, 50, 75, 100, 150, 200, 250):
    r = analyse(ra_c, injected(ra_c, float(P), 1.0, ra_c.t[0] + P * np.arange(N_PHASES) / N_PHASES), float(P))
    print('  %d d %.0f %%' % (P, 100 * np.mean(np.abs(r['P'] - P) / P <= 0.05)), end='')
print()
summary['Colwell et al. (2025)'] = ('0.2 m/s at 50 d', '%.2f m/s at 50 d (50 %% recovered)' % limit(K_C, np.array(rec_C), 50))

## Gómez-Vargas et al. (2026)
P 10–550 d, 10 realisations (random phases) per (P, K), power within ±5 % of the injected period above the analytic
0.1 % FAP in ≥ 70 % of them.

In [ ]:
rng = np.random.default_rng(SEED)
K_G = np.arange(0.05, 0.5 + 1e-9, 0.025)
rate_G, fp_G = {}, {}
for key, ra in models.items():
    thr = ra.power_level(FAP)
    fp_G[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])
    rate = np.empty((len(K_G), len(P_Z)))
    for j, P in enumerate(P_Z):
        t0 = ra.t[0] + P * rng.random(10)
        for i, K in enumerate(K_G):
            rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
    rate_G[key] = rate
    lim = [limit(K_G, rate[:, j], 70) for j in range(len(P_Z))]
    print('%-30s K at 70 %% (* = the series without planet already passes: not usable):' % ra.name)
    print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_G[key])))
lims = np.array([limit(K_G, rate_G['res'][:, j], 70) for j in range(len(P_Z))])[~fp_G['res']]
summary['Gómez-Vargas et al. (2026)'] = ('>= 0.25 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)' % (
    np.nanmin(lims), np.nanmax(lims), len(lims), len(P_Z)))

## Liang et al. (2026)
500 trials (P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase). The top candidate (highest peak) of each
trial is accepted if its power exceeds a threshold; it is a recovery if its period is within 20 % of the injected one
and |K_rec − K|/K < 1, otherwise a false positive. As in the paper, the threshold is the lowest one that gives no
false positive among the 500 trials. The frequency grid goes down to 2.2 d here.

In [ ]:
rng = np.random.default_rng(SEED)
n_tr = 500
P_L = np.exp(rng.uniform(np.log(2.5), np.log(400), n_tr))
K_L = rng.uniform(0.1, 0.7, n_tr)
base = models['res']
ra_l = ResidualAnalysis(base.t, base.y, groups=base.groups, fend=1 / 2.2, name='HARPS-N - SunSim')
t0 = ra_l.t[0] + P_L * rng.random(n_tr)
r = analyse(ra_l, injected(ra_l, P_L, K_L, t0), P_L)
good = (np.abs(r['P'] - P_L) / P_L <= 0.2) & (np.abs(r['K'] - K_L) / K_L < 1)
thr = r['pmax'][~good].max() if (~good).any() else 0                   # lowest threshold with no false positive
found = good & (r['pmax'] > thr)
print('threshold power %.4f (analytic FAP %.1e)' % (thr, ra_l.fap(thr)))
print('recovered %d / %d, of which %d with K < 0.3 m/s' % (found.sum(), n_tr, (found & (K_L < 0.3)).sum()))
bins = np.arange(0.1, 0.71, 0.1)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (K_L >= lo) & (K_L < hi)
    print('  K %.1f-%.1f m/s: %3.0f %% recovered (%d trials)' % (lo, hi, 100 * found[m].mean(), m.sum()))
summary['Liang et al. (2026)'] = ('238 / 500 recovered, 13 with K < 0.3 m/s (NEID, AESTRA)',
                                  '%d / 500 recovered, %d with K < 0.3 m/s' % (found.sum(), (found & (K_L < 0.3)).sum()))

## Summary
Comparable: Zhao et al. and Gómez-Vargas et al. use the same HARPS-N solar data (daily), so the comparison with their
criterion is direct, but their criteria let the series without planet pass at some periods (marked * above), which our
empirical FAP does not. Colwell et al. use the exposures of 2015–2018 only and no FAP: only their 50 d test can be
repeated. Liang et al. use NEID (2020–2024): only the statistics of the 500 trials can be compared, not the data.

In [ ]:
COMPARABLE = {'Ours': '', 'Zhao et al. (2024)': 'same data; their criterion has false positives (*)',
              'Colwell et al. (2025)': 'same instrument, 3 yr of exposures, no FAP',
              'Gómez-Vargas et al. (2026)': 'same data (YARARA spectra); false positives (*)',
              'Liang et al. (2026)': 'NEID, other data: statistics only'}
print('%-28s %-58s %-70s %s' % ('criterion of', 'their result', models['res'].name, 'comparison'))
for name, (theirs, ours) in summary.items():
    print('%-28s %-58s %-70s %s' % (name, theirs, ours, COMPARABLE.get(name, '')))